# Prompt 4: Shuffling and Performance Impact
**Databricks Certified Associate Developer for Apache Spark — Topic 1 (20%)**

---

## Overview

Shuffling is one of the most expensive operations in Spark. Understanding when it happens, why it is costly, and how to minimise it is critical for both the exam and real-world performance tuning.

| Concept | Key idea |
|---|---|
| **Shuffle** | Redistributing data across partitions/nodes over the network |
| **Wide transformation** | Any transform that requires a shuffle (groupBy, join, distinct, orderBy) |
| **Stage boundary** | DAGScheduler splits the plan at every shuffle into a new stage |
| **Shuffle write/read** | Before a shuffle: tasks write shuffle files; after: tasks read from peers |

---
## 1. What is a Shuffle?

A **shuffle** is the process of **repartitioning data across executors** so that rows with the same key end up in the same partition. It involves:

1. **Shuffle write** (map side): each task serialises its output records into shuffle files on local disk, bucketed by the target partition key hash.
2. **Network transfer**: the next stage's tasks fetch (pull) the shuffle blocks they need from the previous stage's executors over the network.
3. **Shuffle read** (reduce side): each task deserialises and merges the blocks it fetched.

```
Stage 1 (map)            Network transfer         Stage 2 (reduce)
─────────────────        ─────────────────        ────────────────────
Executor 1, Task A  ──►  shuffle block 0,1,2  ──► Executor 3, Task X
Executor 2, Task B  ──►  shuffle block 0,1,2  ──► Executor 4, Task Y
                    ──►  shuffle block 0,1,2  ──► Executor 5, Task Z
```

### Why shuffles are expensive

| Cost | Explanation |
|---|---|
| **Disk I/O** | Shuffle files are written to local disk before being read |
| **Network I/O** | Every executor potentially talks to every other executor |
| **Serialisation** | Data is serialised to bytes for disk/network and deserialised on the other side |
| **Memory pressure** | Shuffle read buffers can cause GC pressure and spills |
| **Stage barrier** | The next stage cannot start until all tasks in the previous stage have finished writing their shuffle files |

---
## 2. Operations That Trigger Shuffles

Any **wide transformation** triggers a shuffle. Spark must redistribute rows by key before it can proceed.

| Operation | Why a shuffle is needed |
|---|---|
| `groupBy().agg()` | All rows for each group key must arrive at the same reducer |
| `join()` (SortMergeJoin) | Both sides must be sorted/partitioned by the join key |
| `distinct()` | Duplicates may exist across many partitions — need a global dedup |
| `orderBy()` / `sort()` | Global sort requires all data to be co-located and sorted across partitions |
| `repartition(n)` | Explicitly redistributes data into n new partitions via shuffle |
| `cube()` / `rollup()` | Internally uses groupBy mechanics |
| `reduceByKey()` (RDD) | Aggregates across partitions |
| `cogroup()` (RDD) | Groups multiple RDDs by key |

### Operations that do NOT shuffle

| Operation | Why no shuffle |
|---|---|
| `filter()` / `where()` | Each partition filtered independently |
| `select()` / `withColumn()` | Row-level projection — no key alignment needed |
| `map()` / `flatMap()` | Row-to-row or row-to-many — no reordering |
| `union()` | Just concatenates partition lists |
| `coalesce(n)` | Reduces partition count without full shuffle (narrow) |
| `BroadcastHashJoin` | Small table broadcast to all executors — no shuffle |

> **Exam tip**: `coalesce(n)` does **not** shuffle; `repartition(n)` **does** shuffle. Use `coalesce` when reducing partitions after a filter to avoid unnecessary shuffle cost.

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("ShuffleDemo")
    .config("spark.sql.shuffle.partitions", "8")   # default 200; reduce for demos
    .getOrCreate()
)
print("SparkSession ready")

In [ ]:
# ── Sample data ────────────────────────────────────────────────────────────
sales_data = [
    ("Alice",   "Electronics", 1200.0, "2024-01-05"),
    ("Bob",     "Electronics",  800.0, "2024-01-06"),
    ("Carol",   "Clothing",     250.0, "2024-01-07"),
    ("David",   "Clothing",     300.0, "2024-01-07"),
    ("Eve",     "Electronics", 1500.0, "2024-01-08"),
    ("Frank",   "Furniture",    600.0, "2024-01-09"),
    ("Grace",   "Furniture",    900.0, "2024-01-10"),
    ("Hank",    "Clothing",     150.0, "2024-01-11"),
    ("Ivy",     "Electronics",  950.0, "2024-01-12"),
    ("Jack",    "Furniture",    450.0, "2024-01-12"),
]
sales_df = spark.createDataFrame(
    sales_data, ["salesperson", "category", "amount", "sale_date"]
)
sales_df.show()

In [ ]:
# ── Shuffle-heavy operations ───────────────────────────────────────────────
from pyspark.sql.functions import sum as spark_sum, avg, count, col

print("=== groupBy — triggers shuffle (wide transform) ===")
grouped = sales_df.groupBy("category").agg(
    count("*").alias("num_sales"),
    spark_sum("amount").alias("total_revenue"),
    avg("amount").alias("avg_sale")
)
grouped.show()
# Spark must shuffle all rows — 'Electronics' rows from all partitions
# must be co-located on the same executor before the aggregation can complete

print("\n=== orderBy — triggers shuffle (global sort) ===")
sorted_df = sales_df.orderBy("amount", ascending=False)
sorted_df.show()
# orderBy requires all data to be globally sorted across all partitions
# This forces a full shuffle even if you only want the top row

print("\n=== distinct — triggers shuffle (global dedup) ===")
categories = sales_df.select("category").distinct()
categories.show()
# Duplicates may span partitions — dedup requires shuffle to group by value

In [ ]:
# ── Visualise stages with explain() ──────────────────────────────────────
# Each Exchange node in the physical plan is a shuffle

print("Physical plan for groupBy (look for 'Exchange' — that is the shuffle):")
sales_df.groupBy("category").agg(spark_sum("amount")).explain()

print("Physical plan for filter (no Exchange — no shuffle):")
sales_df.filter(col("amount") > 500).explain()

---
## 3. Performance Implications of Shuffles

### Metrics to watch in the Spark UI

| Metric | Where to find it | What it means |
|---|---|---|
| **Shuffle write size** | Stage detail → Shuffle Write | Total bytes written to shuffle files |
| **Shuffle read size** | Stage detail → Shuffle Read | Total bytes fetched by the next stage |
| **Spill (memory)** | Stage detail → Spill (Memory) | Bytes that couldn't fit in shuffle buffer |
| **Spill (disk)** | Stage detail → Spill (Disk) | Bytes spilled from memory to disk |
| **GC time** | Executor tab → GC Time | Time spent in garbage collection |

### Data skew — the silent killer

A shuffle can be fast on average but one task may receive vastly more data than others (**skew**), causing:
- The job waits for the one slow (skewed) task
- That executor may OOM or spill heavily
- Other executors sit idle

```
Partition 0: 1,000 rows   ← fast task
Partition 1:   900 rows   ← fast task
Partition 2:  50,000 rows ← SLOW / spills to disk  ← job is blocked waiting here
Partition 3:   800 rows   ← fast task (idle)
```

### spark.sql.shuffle.partitions

```python
# Default: 200 — often too high for small data, too low for large data
spark.conf.set("spark.sql.shuffle.partitions", "200")   # default

# Rule of thumb: target ~128 MB per shuffle partition
# If total shuffle data is 10 GB: 10,000 MB / 128 MB ≈ 80 partitions
spark.conf.set("spark.sql.shuffle.partitions", "80")

# With AQE enabled (Spark 3+), Spark auto-coalesces shuffle partitions
spark.conf.set("spark.sql.adaptive.enabled", "true")
```

---
## 4. How to Minimise Shuffling

### Strategy 1: Use broadcast joins for small tables

When one side of a join is small enough to fit in executor memory, broadcast it to avoid shuffling the large table.

```python
from pyspark.sql.functions import broadcast

# SortMergeJoin (default) — both sides shuffled and sorted
result = large_df.join(small_df, on="id")              # shuffle!

# BroadcastHashJoin — small_df sent to all executors; large_df not shuffled
result = large_df.join(broadcast(small_df), on="id")   # no shuffle!
```

Auto-broadcast threshold (default 10 MB):
```python
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "20971520")  # 20 MB
```

### Strategy 2: Filter before join/groupBy

```python
# BAD: join millions of rows then filter
result = df1.join(df2, "id").filter(col("status") == "active")

# BETTER: filter first to reduce rows before the shuffle
result = df1.filter(col("status") == "active").join(df2, "id")
# Catalyst may do this automatically (predicate pushdown) but being explicit is safer
```

### Strategy 3: coalesce() instead of repartition() when reducing

```python
# repartition(n) — always shuffles, good for increasing partitions or rebalancing
df.repartition(50)

# coalesce(n) — merges partitions on same executor when possible; no full shuffle
df.filter(col("active") == True).coalesce(10)   # reduce partitions after filter
```

### Strategy 4: Avoid global orderBy — use window functions or sort within partitions

```python
# BAD: global sort (huge shuffle)
df.orderBy("date")   # all data shuffled into 1 globally sorted partition

# BETTER for per-group ranking: window function (partitioned sort, smaller shuffles)
from pyspark.sql import Window
from pyspark.sql.functions import rank
w = Window.partitionBy("dept").orderBy(col("salary").desc())
df.withColumn("rank", rank().over(w))
```

### Strategy 5: Use AQE (Spark 3+)

```python
spark.conf.set("spark.sql.adaptive.enabled", "true")                  # default true in 3.2+
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")  # merge small partitions
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")           # split skewed partitions
```

### Strategy 6: Pre-partition large tables at write time

```python
# Write partitioned by join key — future joins on that key skip the shuffle
df.write.partitionBy("country").parquet("/data/customers")
# Later joins on 'country' benefit from partition pruning and reduced shuffle
```

In [ ]:
# ── Shuffle-heavy join vs broadcast join ──────────────────────────────────
from pyspark.sql.functions import broadcast

# Large-ish table (simulated)
orders_data = [
    (1, "C001", 500.0),
    (2, "C002", 300.0),
    (3, "C001", 750.0),
    (4, "C003", 200.0),
    (5, "C002", 900.0),
]
orders_df = spark.createDataFrame(orders_data, ["order_id", "cust_id", "total"])

# Small lookup table
customers_data = [
    ("C001", "Alice",   "London"),
    ("C002", "Bob",     "Paris"),
    ("C003", "Carol",   "Berlin"),
]
customers_df = spark.createDataFrame(customers_data, ["cust_id", "name", "city"])

print("=== SortMergeJoin (default) — both sides shuffled ===")
smj = orders_df.join(customers_df, on="cust_id", how="inner")
smj.explain()
smj.show()

print("\n=== BroadcastHashJoin — customers broadcast, no shuffle ===")
bhj = orders_df.join(broadcast(customers_df), on="cust_id", how="inner")
bhj.explain()   # look for 'BroadcastHashJoin' instead of 'SortMergeJoin'
bhj.show()

In [ ]:
# ── repartition vs coalesce ───────────────────────────────────────────────
print(f"Original partitions: {sales_df.rdd.getNumPartitions()}")

# repartition — full shuffle, use when increasing partitions or rebalancing
repart_df = sales_df.repartition(6)
print(f"After repartition(6): {repart_df.rdd.getNumPartitions()} partitions")
print("Plan for repartition (look for Exchange/RoundRobinPartitioning):")
repart_df.explain()

# coalesce — no full shuffle, use only when reducing partitions
coalesced_df = sales_df.coalesce(2)
print(f"\nAfter coalesce(2): {coalesced_df.rdd.getNumPartitions()} partitions")
print("Plan for coalesce (no Exchange — no shuffle):")
coalesced_df.explain()

In [ ]:
# ── spark.sql.shuffle.partitions tuning ──────────────────────────────────
print(f"Current shuffle.partitions: {spark.conf.get('spark.sql.shuffle.partitions')}")

# With default 200 for tiny data — too many empty partitions
spark.conf.set("spark.sql.shuffle.partitions", "200")
result_200 = sales_df.groupBy("category").agg(spark_sum("amount"))
print(f"\nResult partitions at 200 shuffle partitions: {result_200.rdd.getNumPartitions()}")

# Reduce to appropriate size for small data
spark.conf.set("spark.sql.shuffle.partitions", "4")
result_4 = sales_df.groupBy("category").agg(spark_sum("amount"))
print(f"Result partitions at 4 shuffle partitions:   {result_4.rdd.getNumPartitions()}")
result_4.show()
# For production: target ~128 MB per shuffle partition
# With AQE (Spark 3.2+): spark.sql.adaptive.enabled=true does this automatically

In [ ]:
# ── Avoiding global orderBy — use window functions for per-group sort ─────
from pyspark.sql import Window
from pyspark.sql.functions import rank, dense_rank

print("=== Expensive: global orderBy (one big shuffle) ===")
expensive = sales_df.orderBy("amount", ascending=False)
expensive.explain()   # Exchange + Sort across all partitions
expensive.show()

print("\n=== Better: per-category rank with window function ===")
window_spec = Window.partitionBy("category").orderBy(col("amount").desc())
ranked = sales_df.withColumn("rank_in_category", rank().over(window_spec))
ranked.orderBy("category", "rank_in_category").show()
# Each partition (category) is sorted independently — smaller, parallelised shuffle

---
## 5. Real-World Scenarios

<details>
<summary>Scenario 1: groupBy on a high-cardinality column causes 200 near-empty shuffle partitions</summary>

**Situation**: An analyst runs a daily aggregation on a 500 MB sales table using `groupBy('customer_id').agg(sum('amount'))`. The job finishes in 4 minutes even though the data is tiny. Investigation in the Spark UI shows 200 shuffle partitions, most with fewer than 100 rows.

**Root cause**: `spark.sql.shuffle.partitions` defaults to 200. For a 500 MB table this creates 200 post-shuffle partitions, most nearly empty, adding scheduler overhead and tiny-task coordination cost.

```python
# Check current setting
print(spark.conf.get("spark.sql.shuffle.partitions"))  # 200

# Fix: set to a smaller value appropriate for data size
# Rule of thumb: total_shuffle_bytes / 128 MB
# 500 MB / 128 MB ≈ 4 partitions
spark.conf.set("spark.sql.shuffle.partitions", "8")

result = (
    spark.read.parquet("/data/sales")
    .groupBy("customer_id")
    .agg(sum("amount").alias("total_spent"))
)
result.write.parquet("/output/customer_totals")

# Or use AQE to let Spark auto-tune at runtime (Spark 3.2+ default)
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
```

**Expected outcome**: Job time drops from 4 minutes to ~25 seconds. Spark UI shows 8 shuffle partitions each with meaningful data.

**Exam sub-topic**: Performance implications of shuffle; `spark.sql.shuffle.partitions` tuning
</details>

<details>
<summary>Scenario 2: Replacing SortMergeJoin with BroadcastHashJoin eliminates shuffle</summary>

**Situation**: A data pipeline joins a 50 GB transactions table with a 5 MB products lookup table. Each run takes 35 minutes. The Spark UI shows the join stage accounts for 30 of those minutes and involves 50 GB of shuffle write.

**Root cause**: Spark defaults to SortMergeJoin, which requires both sides to be shuffled and sorted by the join key — even when one side is tiny.

```python
from pyspark.sql.functions import broadcast

transactions = spark.read.parquet("/data/transactions")   # 50 GB
products     = spark.read.parquet("/data/products")       # 5 MB

# BAD: SortMergeJoin — both sides shuffled
result_slow = transactions.join(products, on="product_id")

# GOOD: BroadcastHashJoin — products copied to all executors, transactions not shuffled
result_fast = transactions.join(broadcast(products), on="product_id")

# Alternatively, raise autoBroadcast threshold
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024))  # 10 MB

result_fast.explain()   # confirm 'BroadcastHashJoin' in physical plan
result_fast.write.parquet("/output/enriched_transactions")
```

**Expected outcome**: Job drops from 35 minutes to ~5 minutes. Shuffle write drops from 50 GB to near zero. Spark UI shows `BroadcastHashJoin`.

**Exam sub-topic**: Avoiding shuffles with broadcast joins; `spark.sql.autoBroadcastJoinThreshold`
</details>

<details>
<summary>Scenario 3: Data skew in a join causes one task to run 100× longer than others</summary>

**Situation**: A clickstream join on `user_id` runs for 2 hours. All 199 tasks in the join stage finish in under 1 minute but one task runs for the entire 2 hours. The UI shows that one partition holds 80% of the data — a small number of extremely active users skew the data.

**Solutions**:

```python
# Option 1: Enable AQE skew join handling (Spark 3+)
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionFactor", "5")    # default
spark.conf.set("spark.sql.adaptive.skewJoin.skewedPartitionThresholdInBytes", "256mb")
# AQE will automatically detect and split skewed partitions at runtime

# Option 2: Salting — add random prefix to scatter skewed keys
import pyspark.sql.functions as F
NUM_BUCKETS = 10

clicks_salted = clicks.withColumn(
    "salted_user_id",
    F.concat(F.col("user_id"), F.lit("_"), (F.rand() * NUM_BUCKETS).cast("int").cast("string"))
)

# Explode user dimension to match all salt values
users_exploded = users.withColumn("salt", F.explode(F.array([F.lit(i) for i in range(NUM_BUCKETS)])))
users_exploded = users_exploded.withColumn(
    "salted_user_id",
    F.concat(F.col("user_id"), F.lit("_"), F.col("salt").cast("string"))
)

result = clicks_salted.join(users_exploded, on="salted_user_id")
# Now the skewed key is split into 10 sub-keys — even partition distribution
```

**Expected outcome**: With AQE skew join or salting, all tasks finish in roughly equal time. Job runtime drops from 2 hours to ~15 minutes.

**Exam sub-topic**: Data skew; AQE skew join handling; shuffle performance
</details>

<details>
<summary>Scenario 4: coalesce() after filter avoids unnecessary shuffle</summary>

**Situation**: A pipeline reads a 10,000-partition Parquet dataset (written with many small files), filters out 95% of the data, then writes the result. The write stage is slow because it produces 10,000 tiny output files.

**Root cause**: After the filter, 10,000 partitions remain but 9,500 are nearly empty. Writing 10,000 tiny files is slow due to file system overhead.

```python
raw = spark.read.parquet("/data/raw_events")          # 10,000 partitions
print(f"Raw partitions: {raw.rdd.getNumPartitions()}")   # 10,000

filtered = raw.filter(raw.event_type == "purchase")   # 95% filtered out
print(f"Filtered partitions: {filtered.rdd.getNumPartitions()}")  # still 10,000!

# BAD: write 10,000 tiny files
filtered.write.parquet("/output/purchases_bad")

# GOOD: coalesce before write — no shuffle, merges nearby partitions
filtered.coalesce(50).write.parquet("/output/purchases_good")
# Result: 50 reasonably-sized files, much faster to write and later read

# NOTE: do NOT use repartition() here — it would shuffle all data unnecessarily
# filtered.repartition(50)  # ← full shuffle — wasteful
```

**Expected outcome**: Write time drops significantly. Output is 50 files instead of 10,000. No shuffle incurred by using `coalesce()`.

**Exam sub-topic**: `coalesce` vs `repartition`; minimising shuffle overhead
</details>

<details>
<summary>Scenario 5: Pre-partitioning a table at write time eliminates shuffle on future joins</summary>

**Situation**: A customer analytics team runs daily joins between a 500 GB `orders` table and a 100 GB `customers` table on `country_code`. Each join takes 45 minutes due to shuffle. The tables are re-read from Parquet every day.

**Solution**: Write both tables partitioned by `country_code` once. Future joins on that key can use partition pruning and exchange-free join strategies.

```python
# One-time write: partition both tables by the join key
orders_df.write \
    .partitionBy("country_code") \
    .mode("overwrite") \
    .parquet("/data/orders_partitioned")

customers_df.write \
    .partitionBy("country_code") \
    .mode("overwrite") \
    .parquet("/data/customers_partitioned")

# Future reads: filter by country_code → only relevant Parquet folders read
uk_orders = spark.read.parquet("/data/orders_partitioned").filter(col("country_code") == "GB")
uk_customers = spark.read.parquet("/data/customers_partitioned").filter(col("country_code") == "GB")

# Join: Spark can use BucketedHashJoin or reduce shuffle because data is co-partitioned
result = uk_orders.join(uk_customers, on="customer_id")
result.explain()   # much less Exchange in plan
result.write.parquet("/output/uk_enriched_orders")

# spark-submit reference:
# spark-submit --master yarn --deploy-mode cluster \
#   --executor-memory 8g --executor-cores 4 --num-executors 20 \
#   --conf spark.sql.adaptive.enabled=true \
#   daily_join.py
```

**Expected outcome**: With partition pruning, each daily run reads only the relevant `country_code` folders. Join shuffle is drastically reduced. Job time drops from 45 minutes to ~8 minutes.

**Exam sub-topic**: Pre-partitioning to avoid shuffle; partition pruning; performance optimisation
</details>

---
## 6. Quick Reference — Shuffle Minimisation Checklist

```
Shuffle optimisation decision tree
─────────────────────────────────────────────────────────────────────
Is one side of the join < autoBroadcastJoinThreshold?  →  use broadcast()
  │
  └─ No → Can you pre-partition both tables by join key?  →  partitionBy() at write
              │
              └─ No → Enable AQE for automatic optimisation

Reducing partition count after a filter?   →  coalesce(n)  (no shuffle)
Increasing partition count or rebalancing? →  repartition(n)  (shuffle)

shuffle.partitions too high for data size? →  set to total_bytes / 128MB
                               OR          →  enable AQE auto-coalesce
```

### Exam Cheat Sheet

| Question | Answer |
|---|---|
| Which operations trigger a shuffle? | `groupBy`, `join` (SortMerge), `distinct`, `orderBy`, `repartition` |
| Which operations do NOT shuffle? | `filter`, `select`, `withColumn`, `union`, `coalesce` |
| How to avoid join shuffle for a small table? | `broadcast(small_df)` or raise `autoBroadcastJoinThreshold` |
| How to reduce partitions without shuffling? | `coalesce(n)` |
| How to rebalance partitions (full shuffle)? | `repartition(n)` |
| Where is a shuffle visible in the plan? | `Exchange` node in `explain()` output |
| What default setting creates too many shuffle partitions? | `spark.sql.shuffle.partitions = 200` |
| How does Spark 3 auto-tune shuffles? | Adaptive Query Execution (AQE) |

---
## 7. Exam Practice Questions

**Q1**: Which of the following operations does NOT trigger a shuffle?
```
A) df.groupBy('dept').agg(sum('salary'))
B) df.orderBy('salary')
C) df.filter(df.salary > 50000)
D) df.distinct()
```
<details><summary>Answer</summary>
**C) `df.filter(df.salary > 50000)`** — filter is a narrow transformation; each partition is filtered independently. All others (groupBy, orderBy, distinct) are wide and require a shuffle.
</details>

---

**Q2**: You want to reduce a 500-partition DataFrame to 20 partitions after a heavy filter has left most partitions nearly empty. You must not trigger a full shuffle. Which method do you use?

<details><summary>Answer</summary>
**`df.coalesce(20)`** — `coalesce` reduces partitions by merging neighbouring partitions on the same executor without a full shuffle. `repartition(20)` would work but triggers a full shuffle unnecessarily.
</details>

---

**Q3**: Your job is joining a 200 GB fact table with a 4 MB dimension table. What is the best strategy to eliminate the shuffle?

<details><summary>Answer</summary>
Use **`broadcast(dimension_df)`** to force a `BroadcastHashJoin`. The 4 MB table is sent to all executors so the 200 GB table does not need to be shuffled. Alternatively, ensure `spark.sql.autoBroadcastJoinThreshold` is set above 4 MB (default is 10 MB so this may happen automatically).
</details>

---

**Q4**: In the physical plan output of `df.explain()`, which node indicates a shuffle boundary?

<details><summary>Answer</summary>
**`Exchange`** — every `Exchange` node in the physical plan represents a shuffle. You will see `Exchange hashpartitioning(...)` for groupBy/join and `Exchange rangepartitioning(...)` for orderBy.
</details>

---

**Q5**: What is the default value of `spark.sql.shuffle.partitions` and why can it be a problem for small datasets?

<details><summary>Answer</summary>
The default is **200**. For small datasets this creates 200 shuffle output partitions, most nearly empty. Scheduling and managing 200 tiny tasks adds significant overhead. The fix is to lower the value (e.g., `spark.conf.set('spark.sql.shuffle.partitions', '8')`) or enable AQE which auto-coalesces shuffle partitions at runtime (`spark.sql.adaptive.enabled=true`).
</details>